# Part 1 : Practise

## Task 1 : Load it, remove the empty columns

### Load the file and make a copy

In [1]:
import pandas as pd

rentals_raw = pd.read_csv("data/bike_rentals.csv")
rentals = rentals_raw.copy()

### print the shape

In [2]:
rentals.shape

(403, 15)

### Find the data type of all columns

In [3]:
rentals.dtypes

rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object

### Find the empty columns

In [4]:
rentals.columns[rentals.isna().all()]

Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')

- isna() puts True wherever a value is missing
- all() asks for each column is every vale True?
- rentals.columns[...] keeps only the column names where the answer was True

### Remove the empty columns

In [5]:
rentals = rentals.dropna(axis=1, how="all")

### dropna()

By default, `dropna()` removes the **rows** that have at least one missing value.

These two lines are the same:

    rentals.dropna()
    rentals.dropna(axis=0, how='any')

- `axis=0` means rows
- `how='any'` means remove the row if even one value is missing

This one removes the **columns** where the whole column is empty:

    rentals.dropna(axis=1, how='all')

- `axis=1` means columns
- `how='all'` means remove the column only if all its values are missing

In [6]:
rentals.shape

(403, 12)

In [7]:
rentals.columns[rentals.isna().all()]

Index([], dtype='str')

**Answer:** 
- The table has 403 rows and 15 columns. 
- The price column is text (str), not a number.
- I found 3 empty columns: coupon_code, damage_notes and late_fee_note. 
- After I removed them, 12 columns are left and 0 columns are empty.

## Task 2 : The price: from text to numbers

### Step 1 : Remove Sek

In [8]:
rentals["price"] = rentals["price"].str.replace("SEK ", "", regex=False)

In [9]:
rentals["price"].head(3)

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str

### Step 2 : Remove the comma

In [10]:
rentals["price"] = rentals["price"].str.replace(",", "", regex=False)

In [11]:
rentals["price"].head(3)

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

### Change the type from str to float

In [12]:
rentals["price"] = rentals["price"].astype(float)

In [13]:
rentals["price"].dtype

dtype('float64')

In [14]:
rentals["price"].describe()

count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64

### Three most expensive rentals for columns (rental_id, bike_type, status and price)


In [15]:
rentals.sort_values("price", ascending=False).head(3)[["rental_id", "bike_type", "status", "price"]]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


**Answer:** 
- The price was text like "SEK 1,311.50". 
- First I removed "SEK " and then the comma.
- After that I changed it to float. 
- 352 rentals have a price. 
- The mean is 847.3, the median is 668.2, the lowest price is 154.5 and the highest is 3401.0. 
- The three most expensive rentals are R-5282, R-5209 and R-5378.

## Task 3 : The missing Prices

### Find how many prices are missing and it's percent

In [16]:
rentals["price"].isna().sum()

np.int64(51)

In [17]:
(rentals["price"].isna().mean()*100).round(1)

np.float64(12.7)

In [18]:
pd.crosstab(rentals["price"].isna(), rentals["status"] == "Cancelled")

status,False,True
price,,
False,352,0
True,4,47


**Answer:** 
- 51 prices are missing, which is 12.7 %.
- Among the 51 missing prices, 47 of them are cancelled rentals, so they are missing for a reason: nothing was paid. 
- 4 are not cancelled, so they are missing by accident.
- I will leave all 51 as missing, because I don't know the real prices. 

## Task 4 : The letters t and f

### Find the columns that contain only t and f

In [19]:
tf_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {"t", "f"}]
tf_cols

['is_member', 'insured', 'paid_online']

In [20]:
rentals["status"].unique()

<StringArray>
['Completed', 'No-show', 'Cancelled']
Length: 3, dtype: str

- "status" is not in the list because it doesn't have t and f values.
- It has 'Completed', 'No-show', 'Cancelled' values 

### Change the columns to True and False with map

In [21]:
for c in tf_cols:
    rentals[c] = rentals[c].map({"t": True, "f": False})

### How many missing values does each column have afterwords

In [22]:
rentals[tf_cols].isna().sum()

is_member      0
insured        3
paid_online    0
dtype: int64

### Compare median Price of members and non-members

In [23]:
rentals["is_member"].value_counts()

is_member
False    288
True     115
Name: count, dtype: int64

In [24]:
rentals.groupby("is_member")["price"].agg(["median", "count"])

,median,count
is_member,,
False,756.25,248
True,558.00,104


**Answer:** 
- The columns with only t and f are is_member, insured and paid_online. 
- status is not in the list because it has the words Completed, Cancelled and No-show. 
- After map, insured has 3 missing values and the other two have 0. 
- There are 288 non-members and 115 members. 
- The median price is 756 for non-members (248 rentals) and 558 for members (104 rentals). 
- Members pay less, but I can't say yet that being a member is the reason.

## Task 5 : The dates

### Check the start_date and end_date column data type

In [25]:
rentals[["start_date", "end_date"]].dtypes

start_date    str
end_date      str
dtype: object

### Count the missing values in both the columns

In [26]:
print(rentals["start_date"].isna().sum())
print(rentals["end_date"].isna().sum())

0
0


### Convert the data type to dates with errors="coerce"

In [27]:
rentals["start_date"] = pd.to_datetime(rentals["start_date"], errors="coerce")

In [28]:
rentals["end_date"] = pd.to_datetime(rentals["end_date"], errors="coerce")

Notes:
**Why errors='coerce'?**

to_datetime makes real calendar dates from the text. It reads the text "2026-02-30" as year 2026, month 02 and day 30 (year-month-day).

It crashes for eg:
- "2026-02-30" : date that does not exist (February has only 28 days in 2026)
- "2026-13-05" : there’s no month 13
- "unknown", "soon", "abc" : it’s not a date at all
Then the whole line stops with an error.

It doesn't crash for eg:
- "2026-04-27" : a normal date 
- an empty cell (NaN) : becomes NaT, no error


With errors='coerce', the bad dates become NaT (missing) and the rest is changed to dates.

### Count the missing values after change the data type

In [29]:
print(rentals["start_date"].isna().sum())
print(rentals["end_date"].isna().sum())

2
0


### Check the data types

In [30]:
rentals[["start_date", "end_date"]].dtypes

start_date    datetime64[us]
end_date      datetime64[us]
dtype: object

In [31]:
rentals["rental_days"] = (rentals["end_date"] - rentals["start_date"]).dt.days

In [32]:
rentals["rental_days"].describe()

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64

**Answer:** 
- Before the change, both date columns had 0 missing values. 
- After to_datetime with errors='coerce', start_date has 2 missing, because 2 dates could not exist on the calendar. end_date still has 0. 
- Both columns are datetime64 now. 
- rental_days has 401 values, the mean is 2.96, the max is 7 and the min is -2. A rental can't end before it starts, so -2 is wrong. 
- Maybe someone typed the dates in the wrong order.

## Task 6 : The missing ratings 

### How many ratings are missing

In [33]:
rentals["rating"].isna().sum()

np.int64(115)

### Compare with status with pd.crosstab

In [34]:
pd.crosstab(rentals["rating"].isna(), rentals["status"])

status,Cancelled,Completed,No-show
rating,,,
False,0,288,0
True,47,46,22


- 115 ratings are missing.
- Among the 115 missing ratings 47 are cancelled and 22 are no-shows so the bike was never ridden, this was the reason for missing rating.
- 46 are completed rentals where the customer rode the bike but chose not to give a rating.
- Not a perfect match. In the lecture, missing = no reviews, every time. Here, completed rentals can also be missing a rating because customers chose not to give a rating

### Create a column and count

In [35]:
rentals["has_rating"] = rentals["rating"].notna()

In [36]:
rentals["has_rating"].value_counts()

has_rating
True     288
False    115
Name: count, dtype: int64

In [37]:
rentals["rating"].describe()

count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

median is 5.0

## Task 7 : The helmets text

In [38]:
rentals["helmets_text"].value_counts(dropna=False)

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64

### Create child_helmet column

In [39]:
rentals["child_helmet"] = rentals["helmets_text"].str.contains("child helmet")

In [40]:
rentals["child_helmet"].sum()

np.int64(47)

In [41]:
rentals["child_helmet"].value_counts(dropna=False)

child_helmet
False    356
True      47
Name: count, dtype: int64

### Create column helmets_count 

In [42]:
rentals["helmets_count"] = rentals["helmets_text"].str.extract(r'(\d+)', expand=False)

In [43]:
rentals["helmets_count"].value_counts(dropna=False)

helmets_count
1      193
2      127
NaN     63
3       20
Name: count, dtype: int64

Here 
- sum of 146 and 47 from 1 helmet and 1 child helmet made 193
- no helemts 59 plus NAN 4 gave NaN as 63

But we should have 59 rows as 0 because it belongs to No helmets and only 4 were missing which should have NaN

### Build a mask to find the rows where helmets_text had value as no helmets

In [44]:
no_helmets = rentals["helmets_text"].str.lower().str.contains("no helmets")
no_helmets.sum()

np.int64(59)

### replace those rows with 0

In [45]:
# where(mask, value) → keeps the value where the mask is True, puts value where it’s False.
# So the mask has to be True on the rows you want to keep, and False on the rows you want to change.

rentals["helmets_count"] = rentals["helmets_count"].where(~no_helmets, 0)

In [46]:
rentals["helmets_count"].value_counts(dropna=False)

helmets_count
1      193
2      127
0       59
3       20
NaN      4
Name: count, dtype: int64

In [47]:
rentals["child_helmet"].value_counts(dropna=False)

child_helmet
False    356
True      47
Name: count, dtype: int64

- child_helmet has 47 rows with True values
- NaN had 63 values as No helmets got as NaN while extracting as there were no numbers before it, so found the rows where the value was no helmets and replaced those 59 rows with 0 instead of NaN
- other 4 were missing so kept as NaN
- helmets_count numbers

    - 0 : 59
    - 1 : 193
    - 2 : 127
    - 3 : 20
    - Nan : 4

## Task 8 : Spellings and duplicates

### Find the no of unique values

In [48]:
rentals[["station", "bike_type", "status"]].nunique()

station      20
bike_type     3
status        3
dtype: int64

### check the station column as it has more unique values

In [49]:
rentals["station"].value_counts(dropna=False)

station
Central         92
Södermalm       85
Kungsholmen     85
Östermalm       54
Hammarby        53
CENTRAL          6
 Central         4
 Södermalm       4
södermalm        4
central          3
Central          2
Östermalm        2
Kungsholmen      2
Hammarby         1
 Kungsholmen     1
SÖDERMALM        1
Södermalm        1
ÖSTERMALM        1
kungsholmen      1
hammarby         1
Name: count, dtype: int64

### loop through those 3 column names and clean by removing space and lower case

In [50]:
for c in ["station", "bike_type", "status"]:
    print(c, rentals[c].str.strip().str.lower().nunique())

station 5
bike_type 3
status 3


station column had problem as after cleaning the number of unique values goes down

### Title case for station

In [51]:
rentals["station"] = rentals["station"].str.strip().str.title()

In [52]:
rentals["station"].value_counts()

station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64

### Check for complete duplicate rows

In [53]:
rentals.duplicated().sum()

np.int64(3)

In [54]:
rentals["rental_id"].duplicated().sum()

np.int64(3)

so there are 3 duplicate rows, all the 3 rows has same id

### Remove the duplicates

In [55]:
rentals = rentals.drop_duplicates().reset_index(drop=True)

In [56]:
rentals.shape

(400, 16)

While starting
- rentals had :  (403, 15) 
- then removed empty columns and got (403, 12)
- after cleaning, adding new columns and removing duplicates got (400, 16)
- 4 new columns (rental_days, has_rating, child_helmet, helmets_count).